# CSMS AI Marine Scoring — Batch Evaluation: All Categories

**Complete end-to-end batch testing across all 29 marine categories**

## What This Notebook Does:
1. Discovers all 29 categories in `test_submissions/marine/`
2. Runs the **original pipeline** (flat aggregation) on each category
3. Runs the **enhanced pipeline** with `text-embedding-3-small` (1536-dim)
4. Optionally runs the **enhanced pipeline** with `text-embedding-3-large` (3072-dim) for comparison
5. Compares expected scores (from `expected_result.json`) vs AI predictions
6. Generates comprehensive HTML dashboard with progress bars and visualizations
7. Saves all results to JSON for downstream analysis

## Configuration:
- **Embedding models:** `text-embedding-3-small` (fast, cheap) vs `text-embedding-3-large` (higher accuracy)
- **Storage:** SQLite vector database with FTS5 full-text search
- **Retrieval:** Hybrid semantic + keyword search via Reciprocal Rank Fusion
- **LLM:** VLLM (Qwen3.5:27b) for jurors and consensus

## Expected Runtime:
- With `text-embedding-3-small`: ~15-20 minutes for all 29 categories
- With `text-embedding-3-large`: ~20-25 minutes (embeddings are more expensive)

---

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import re
import time
import hashlib
import struct
import sqlite3
import logging
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, HTML, clear_output
from tqdm.notebook import tqdm

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

PROJECT_ROOT = os.path.abspath('..')
MARINE_DIR = os.path.join(PROJECT_ROOT, 'test_submissions', 'marine')
KB_DIR = os.path.join(PROJECT_ROOT, 'knowledge_base', 'marine')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'notebook_v2', 'outputs')
DB_DIR = os.path.join(OUTPUT_DIR, 'databases')
REPORT_DIR = os.path.join(OUTPUT_DIR, 'reports')

os.makedirs(DB_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY not found in environment. Check .env file.')

openai_client = OpenAI(api_key=OPENAI_API_KEY)

EMBEDDING_MODEL_SMALL = 'text-embedding-3-small'
EMBEDDING_MODEL_LARGE = 'text-embedding-3-large'
EMBEDDING_DIM_SMALL = 1536
EMBEDDING_DIM_LARGE = 3072

CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
TOP_K = 15

JUROR_ROLE = 'reasoning_model'
CONSENSUS_ROLE = 'consensus_model'

print(f'Project root: {PROJECT_ROOT}')
print(f'Marine dir:   {MARINE_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'API key:      {"loaded" if OPENAI_API_KEY else "MISSING"}')
print(f'\nEmbedding models:')
print(f'  Small: {EMBEDDING_MODEL_SMALL} ({EMBEDDING_DIM_SMALL}-dim)')
print(f'  Large: {EMBEDDING_MODEL_LARGE} ({EMBEDDING_DIM_LARGE}-dim)')
print('\nSetup complete.')

## 2. Configuration

In [ ]:
RUN_ORIGINAL_PIPELINE = True
RUN_ENHANCED_SMALL = False
RUN_ENHANCED_LARGE = True

PARALLEL_CATEGORIES = 1
SAVE_INTERMEDIATE_RESULTS = True
VERBOSE = False

print('=== Batch Configuration ===')
print(f'Run original pipeline:     {RUN_ORIGINAL_PIPELINE}')
print(f'Run enhanced (small):      {RUN_ENHANCED_SMALL}')
print(f'Run enhanced (large):      {RUN_ENHANCED_LARGE}')
print(f'Parallel categories:       {PARALLEL_CATEGORIES}')
print(f'Save intermediate:         {SAVE_INTERMEDIATE_RESULTS}')
print(f'Verbose output:            {VERBOSE}')

## 3. Category Discovery

In [ ]:
def discover_categories(marine_dir: str) -> List[Dict[str, Any]]:
    categories = []
    pattern = re.compile(r'^([0-9]+(\.[0-9]+)?)')
    
    for item in sorted(os.listdir(marine_dir)):
        full_path = os.path.join(marine_dir, item)
        if os.path.isdir(full_path):
            m = pattern.match(item)
            if m:
                cat_id = m.group(1)
                
                expected_file = os.path.join(full_path, 'expected_result.json')
                expected_score = None
                if os.path.exists(expected_file):
                    with open(expected_file, 'r', encoding='utf-8') as f:
                        expected_score = json.load(f).get('expected_score')
                
                categories.append({
                    'id': cat_id,
                    'name': item,
                    'path': full_path,
                    'expected_score': expected_score
                })
    
    def sort_key(x):
        parts = x['id'].split('.')
        return [int(p) for p in parts]
    
    categories.sort(key=sort_key)
    return categories

categories = discover_categories(MARINE_DIR)

print(f'=== Discovered {len(categories)} Categories ===\n')

df_cats = pd.DataFrame([
    {'#': i+1, 'ID': c['id'], 'Name': c['name'], 'Expected': c['expected_score']}
    for i, c in enumerate(categories)
])
display(df_cats)

total_expected = sum(c['expected_score'] for c in categories if c['expected_score'] is not None)
print(f'\nTotal expected score: {total_expected} / {len(categories) * 10} (if all were 10)')

## 4. Core Functions

In [ ]:
from engine.extractors.router import ExtractorRouter
from engine.aggregators.document_classifier import DocumentClassifier
from engine.aggregators.priority_assigner import PriorityAssigner
from engine.aggregators.evidence_aggregator import EvidenceAggregator
from engine.validators.engine import ValidatorEngine
from engine.jurors.compliance_juror import ComplianceJuror
from engine.jurors.evidence_juror import EvidenceJuror
from engine.jurors.hse_juror import HSEJuror
from engine.consensus.consensus_agent import ConsensusAgent

def snap_to_csms_scale(val):
    try:
        v = float(val)
    except Exception:
        return 6
    if v < 1.5: return 0
    if v < 4.5: return 3
    if v < 8.0: return 6
    return 10

def load_kb(category_id: str) -> Dict[str, Any]:
    clean_id = re.match(r'^[0-9]+(\.[0-9]+)?', str(category_id).strip())
    clean_id = clean_id.group(0) if clean_id else str(category_id).strip()
    result = {}
    for kind, sub, prefix in [('category', 'categories', ''), ('scoring_rules', 'scoring_rules', 'SR-'), ('validation_rules', 'validation_rules', 'VR-')]:
        fpath = os.path.join(KB_DIR, sub, f'{prefix}{clean_id}.json')
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                result[kind] = json.load(f)
        else:
            result[kind] = {}
    return result

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if not text or not text.strip():
        return []
    text = text.strip()
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        if end >= len(text):
            break
        start = end - overlap
    return chunks

def create_chunks_from_docs(extracted_docs: List[Dict[str, Any]], category_id: str) -> List[Dict[str, Any]]:
    all_chunks = []
    for doc in extracted_docs:
        filename = doc.get('file', 'unknown')
        doc_type = doc.get('document_type', 'unknown')
        tier = doc.get('classification', 'Unclassified')
        priority = doc.get('priority', 4)
        
        evidence_list = doc.get('evidence', [])
        for ev_idx, evidence in enumerate(evidence_list):
            ev_str = str(evidence)
            if not ev_str.strip():
                continue
            text_chunks = chunk_text(ev_str)
            for ch_idx, chunk_text_val in enumerate(text_chunks):
                chunk_id = hashlib.sha256(f'{category_id}|{filename}|{ev_idx}|{ch_idx}'.encode()).hexdigest()[:16]
                all_chunks.append({
                    'chunk_id': chunk_id,
                    'category_id': category_id,
                    'filename': filename,
                    'doc_type': doc_type,
                    'tier': tier,
                    'priority': priority,
                    'evidence_idx': ev_idx,
                    'chunk_idx': ch_idx,
                    'text': chunk_text_val,
                    'char_count': len(chunk_text_val)
                })
    return all_chunks

def embed_texts(texts: List[str], model: str = EMBEDDING_MODEL_SMALL, batch_size: int = 100) -> List[List[float]]:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        cleaned = [t.replace('\n', ' ').strip() or 'empty' for t in batch]
        try:
            response = openai_client.embeddings.create(model=model, input=cleaned)
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
        except Exception as e:
            logger.error(f'Embedding error: {e}')
            dim = EMBEDDING_DIM_SMALL if 'small' in model else EMBEDDING_DIM_LARGE
            all_embeddings.extend([[0.0] * dim] * len(batch))
    return all_embeddings

def init_evidence_db(db_path: str, embedding_dim: int) -> sqlite3.Connection:
    if os.path.exists(db_path):
        os.remove(db_path)
    conn = sqlite3.connect(db_path)
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA synchronous=NORMAL')
    
    conn.execute(f'''
        CREATE TABLE IF NOT EXISTS chunks (
            chunk_id TEXT PRIMARY KEY,
            category_id TEXT NOT NULL,
            filename TEXT NOT NULL,
            doc_type TEXT,
            tier TEXT,
            priority INTEGER,
            evidence_idx INTEGER,
            chunk_idx INTEGER,
            text TEXT NOT NULL,
            char_count INTEGER,
            embedding BLOB,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    conn.execute('CREATE INDEX IF NOT EXISTS idx_category ON chunks(category_id)')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_filename ON chunks(filename)')
    
    conn.execute('''
        CREATE VIRTUAL TABLE IF NOT EXISTS chunks_fts USING fts5(
            chunk_id, text, filename, category_id,
            content='chunks',
            content_rowid='rowid'
        )
    ''')
    
    conn.execute('''
        CREATE TRIGGER IF NOT EXISTS chunks_ai AFTER INSERT ON chunks BEGIN
            INSERT INTO chunks_fts(chunk_id, text, filename, category_id)
            VALUES (NEW.chunk_id, NEW.text, NEW.filename, NEW.category_id);
        END
    ''')
    
    conn.commit()
    return conn

def embedding_to_blob(embedding: List[float]) -> bytes:
    return struct.pack(f'{len(embedding)}f', *embedding)

def blob_to_embedding(blob: bytes) -> List[float]:
    n = len(blob) // 4
    return list(struct.unpack(f'{n}f', blob))

def store_chunks_in_db(conn: sqlite3.Connection, chunks: List[Dict[str, Any]]) -> int:
    cursor = conn.cursor()
    inserted = 0
    for ch in chunks:
        emb_blob = embedding_to_blob(ch['embedding']) if ch.get('embedding') else None
        try:
            cursor.execute('''
                INSERT OR REPLACE INTO chunks 
                (chunk_id, category_id, filename, doc_type, tier, priority, evidence_idx, chunk_idx, text, char_count, embedding)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                ch['chunk_id'], ch['category_id'], ch['filename'], ch['doc_type'],
                ch['tier'], ch['priority'], ch['evidence_idx'], ch['chunk_idx'],
                ch['text'], ch['char_count'], emb_blob
            ))
            inserted += 1
        except Exception as e:
            logger.error(f'DB insert error: {e}')
    conn.commit()
    return inserted

def build_query_from_kb(kb: Dict[str, Any]) -> str:
    parts = []
    cat = kb.get('category', {})
    sr = kb.get('scoring_rules', {})
    vr = kb.get('validation_rules', {})
    
    if cat.get('category_name'):
        parts.append(f'Category: {cat["category_name"]}')
    if cat.get('required_evidence'):
        parts.append(f'Required evidence: {", ".join(cat["required_evidence"])}')
    if sr.get('reasoning_hint'):
        parts.append(f'Scoring criteria: {sr["reasoning_hint"]}')
    if vr.get('ai_checkpoint_keywords'):
        parts.append(f'Keywords: {", ".join(vr["ai_checkpoint_keywords"])}')
    if vr.get('mandatory_fields'):
        parts.append(f'Mandatory: {", ".join(vr["mandatory_fields"])}')
    
    return '\n'.join(parts)

def semantic_search(conn: sqlite3.Connection, query_embedding: List[float], category_id: str = None, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    query = 'SELECT chunk_id, category_id, filename, doc_type, tier, priority, text, char_count, embedding FROM chunks'
    params = []
    if category_id:
        query += ' WHERE category_id = ?'
        params.append(category_id)
    
    rows = conn.execute(query, params).fetchall()
    if not rows:
        return []
    
    query_vec = np.array(query_embedding, dtype=np.float32)
    query_norm = np.linalg.norm(query_vec)
    if query_norm == 0:
        return []
    
    scored = []
    for row in rows:
        chunk_id, cat_id, filename, doc_type, tier, priority, text, char_count, emb_blob = row
        if emb_blob is None:
            continue
        chunk_vec = np.array(blob_to_embedding(emb_blob), dtype=np.float32)
        chunk_norm = np.linalg.norm(chunk_vec)
        if chunk_norm == 0:
            continue
        cosine_sim = float(np.dot(query_vec, chunk_vec) / (query_norm * chunk_norm))
        scored.append({
            'chunk_id': chunk_id,
            'category_id': cat_id,
            'filename': filename,
            'doc_type': doc_type,
            'tier': tier,
            'priority': priority,
            'text': text,
            'char_count': char_count,
            'semantic_score': round(cosine_sim, 6)
        })
    
    scored.sort(key=lambda x: x['semantic_score'], reverse=True)
    return scored[:top_k]

def keyword_search(conn: sqlite3.Connection, keywords: List[str], category_id: str = None, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    if not keywords:
        return []
    query_str = ' OR '.join(keywords)
    sql = '''
        SELECT c.chunk_id, c.category_id, c.filename, c.doc_type, c.tier, c.priority, c.text, c.char_count
        FROM chunks_fts fts
        JOIN chunks c ON c.chunk_id = fts.chunk_id
        WHERE chunks_fts MATCH ?
    '''
    params = [query_str]
    if category_id:
        sql += ' AND fts.category_id = ?'
        params.append(category_id)
    sql += f' LIMIT {top_k * 2}'
    
    try:
        rows = conn.execute(sql, params).fetchall()
    except Exception as e:
        logger.error(f'FTS5 search error: {e}')
        return []
    
    results = []
    for i, row in enumerate(rows):
        results.append({
            'chunk_id': row[0], 'category_id': row[1], 'filename': row[2],
            'doc_type': row[3], 'tier': row[4], 'priority': row[5],
            'text': row[6], 'char_count': row[7],
            'keyword_rank': i + 1
        })
    return results

def hybrid_search(conn: sqlite3.Connection, query_text: str, keywords: List[str], category_id: str = None, top_k: int = TOP_K, embedding_model: str = EMBEDDING_MODEL_SMALL, k_rrf: int = 60) -> List[Dict[str, Any]]:
    query_emb = embed_texts([query_text], model=embedding_model)[0]
    
    semantic_results = semantic_search(conn, query_emb, category_id, top_k=top_k * 2)
    keyword_results = keyword_search(conn, keywords, category_id, top_k=top_k * 2)
    
    scores = {}
    for rank, item in enumerate(semantic_results):
        cid = item['chunk_id']
        scores[cid] = scores.get(cid, {'data': item, 'rrf': 0.0})
        scores[cid]['rrf'] += 1.0 / (k_rrf + rank + 1)
    
    for rank, item in enumerate(keyword_results):
        cid = item['chunk_id']
        if cid not in scores:
            scores[cid] = {'data': item, 'rrf': 0.0}
        scores[cid]['rrf'] += 1.0 / (k_rrf + rank + 1)
    
    ranked = sorted(scores.values(), key=lambda x: x['rrf'], reverse=True)
    
    results = []
    for item in ranked[:top_k]:
        entry = item['data'].copy()
        entry['rrf_score'] = round(item['rrf'], 6)
        results.append(entry)
    
    return results

def build_enhanced_aggregation(relevant_chunks: List[Dict[str, Any]], kb: Dict[str, Any], use_openai: bool = False) -> Dict[str, Any]:
    if not relevant_chunks:
        return {'key_findings': [], 'document_types_found': [], 'missing_critical_info': True, 'source_chunks': 0}
    
    chunk_texts = [f'[{ch["filename"]}] {ch["text"]}' for ch in relevant_chunks]
    combined = '\n\n'.join(chunk_texts)
    
    if len(combined) > 12000:
        combined = combined[:12000] + '\n...[truncated]'
    
    if use_openai:
        schema = {
            'type': 'object',
            'properties': {
                'key_findings': {'type': 'array', 'items': {'type': 'string'}},
                'document_types_found': {'type': 'array', 'items': {'type': 'string'}},
                'missing_critical_info': {'type': 'boolean'}
            },
            'required': ['key_findings', 'document_types_found', 'missing_critical_info']
        }
        
        scoring_hint = kb.get('scoring_rules', {}).get('reasoning_hint', '')
        mandatory = kb.get('validation_rules', {}).get('mandatory_fields', [])
        
        prompt = f'''Aggregate the following semantically-retrieved evidence chunks for CSMS category evaluation.

Scoring criteria hint: {scoring_hint}
Mandatory fields to check: {mandatory}

Retrieved Evidence ({len(relevant_chunks)} chunks):
{combined}

Identify key findings, document types found, and whether critical information is missing.'''
        
        system_prompt = 'You are the Evidence Aggregator Agent. Summarize retrieved evidence precisely. Focus on compliance-relevant facts.'
        
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt + f'\n\nRespond with JSON: {json.dumps(schema)}'}
            ]
            response = openai_client.chat.completions.create(
                model='gpt-4o-mini',
                messages=messages,
                temperature=0.0,
                response_format={'type': 'json_object'}
            )
            content = response.choices[0].message.content.strip()
            if content.startswith('```json'):
                content = content[7:]
            if content.startswith('```'):
                content = content[3:]
            if content.endswith('```'):
                content = content[:-3]
            result = json.loads(content.strip())
        except Exception as e:
            logger.error(f'LLM aggregation error: {e}')
            result = {
                'key_findings': [ch['text'][:200] for ch in relevant_chunks[:10]],
                'document_types_found': list(set(ch.get('doc_type', 'unknown') for ch in relevant_chunks)),
                'missing_critical_info': False
            }
    else:
        result = {
            'key_findings': [ch['text'][:200] for ch in relevant_chunks[:10]],
            'document_types_found': list(set(ch.get('doc_type', 'unknown') for ch in relevant_chunks)),
            'missing_critical_info': False
        }
    
    result['source_chunks'] = len(relevant_chunks)
    result['retrieval_method'] = 'hybrid_semantic_keyword'
    return result

print('Core functions loaded.')

## 5. Pipeline Functions

In [ ]:
IGNORE_FILES = {'expected_result.json', 'readme.md', '.keep', '.placeholder', 'ignore_me.txt', 'thumbs.db', '.ds_store'}

def extract_documents(folder_path: str) -> List[Dict[str, Any]]:
    extractor = ExtractorRouter()
    classifier = DocumentClassifier()
    assigner = PriorityAssigner()
    
    extracted = []
    for root, _, files in os.walk(folder_path):
        for file in sorted(files):
            if file.lower() in IGNORE_FILES:
                continue
            fpath = os.path.join(root, file)
            try:
                doc_data = extractor.extract_document(fpath)
                if doc_data.get('status') != 'error':
                    doc_data['file'] = file
                    cls = classifier.classify(fpath, doc_data.get('metadata', {}))
                    doc_data['classification'] = cls
                    doc_data['priority'] = assigner.assign_priority(cls)
                    extracted.append(doc_data)
            except Exception as e:
                logger.warning(f'Extraction error for {file}: {e}')
    
    return extracted

def run_original_pipeline(category_id: str, folder_path: str, kb: Dict[str, Any]) -> Dict[str, Any]:
    t_start = time.time()
    
    extracted = extract_documents(folder_path)
    
    aggregator = EvidenceAggregator()
    aggregated = aggregator.aggregate(extracted)
    
    validator = ValidatorEngine()
    val_methods = kb.get('validation_rules', {}).get('validation_methods', [])
    ev_list = [str(d.get('evidence', [])) for d in extracted]
    val_results = validator.run_validations(val_methods, {'evidence': ev_list}, kb['validation_rules'])
    
    comp_j = ComplianceJuror(role=JUROR_ROLE)
    ev_j = EvidenceJuror(role=JUROR_ROLE)
    hse_j = HSEJuror(role=JUROR_ROLE)
    
    comp_r = comp_j.evaluate(kb, aggregated, val_results)
    ev_r = ev_j.evaluate(kb, aggregated, val_results)
    hse_r = hse_j.evaluate(kb, aggregated, val_results)
    
    consensus = ConsensusAgent(role=CONSENSUS_ROLE)
    final = consensus.resolve(comp_r, ev_r, hse_r, kb)
    recommended = snap_to_csms_scale(final.get('recommended_score', 6))
    
    elapsed = round(time.time() - t_start, 2)
    
    return {
        'predicted': recommended,
        'confidence': final.get('confidence', 0.0),
        'comp_score': snap_to_csms_scale(comp_r.get('score', 6)),
        'ev_score': snap_to_csms_scale(ev_r.get('score', 6)),
        'hse_score': snap_to_csms_scale(hse_r.get('score', 6)),
        'time': elapsed,
        'docs': len(extracted)
    }

def run_enhanced_pipeline(category_id: str, folder_path: str, kb: Dict[str, Any], embedding_model: str, embedding_dim: int, db_path: str, use_openai_agg: bool = False) -> Dict[str, Any]:
    t_start = time.time()
    
    extracted = extract_documents(folder_path)
    
    chunks = create_chunks_from_docs(extracted, category_id)
    
    if chunks:
        texts = [c['text'] for c in chunks]
        embs = embed_texts(texts, model=embedding_model)
        for i, emb in enumerate(embs):
            chunks[i]['embedding'] = emb
        
        conn = init_evidence_db(db_path, embedding_dim)
        store_chunks_in_db(conn, chunks)
    else:
        conn = None
    
    if conn and chunks:
        query_text = build_query_from_kb(kb)
        keywords = kb.get('validation_rules', {}).get('ai_checkpoint_keywords', [])
        relevant = hybrid_search(conn, query_text, keywords, category_id=category_id, top_k=TOP_K, embedding_model=embedding_model)
        conn.close()
    else:
        relevant = []
    
    aggregated = build_enhanced_aggregation(relevant, kb, use_openai=use_openai_agg)
    
    validator = ValidatorEngine()
    val_methods = kb.get('validation_rules', {}).get('validation_methods', [])
    ev_list = [str(d.get('evidence', [])) for d in extracted]
    val_results = validator.run_validations(val_methods, {'evidence': ev_list}, kb['validation_rules'])
    
    comp_j = ComplianceJuror(role=JUROR_ROLE)
    ev_j = EvidenceJuror(role=JUROR_ROLE)
    hse_j = HSEJuror(role=JUROR_ROLE)
    
    comp_r = comp_j.evaluate(kb, aggregated, val_results)
    ev_r = ev_j.evaluate(kb, aggregated, val_results)
    hse_r = hse_j.evaluate(kb, aggregated, val_results)
    
    consensus = ConsensusAgent(role=CONSENSUS_ROLE)
    final = consensus.resolve(comp_r, ev_r, hse_r, kb)
    recommended = snap_to_csms_scale(final.get('recommended_score', 6))
    
    elapsed = round(time.time() - t_start, 2)
    
    return {
        'predicted': recommended,
        'confidence': final.get('confidence', 0.0),
        'comp_score': snap_to_csms_scale(comp_r.get('score', 6)),
        'ev_score': snap_to_csms_scale(ev_r.get('score', 6)),
        'hse_score': snap_to_csms_scale(hse_r.get('score', 6)),
        'time': elapsed,
        'docs': len(extracted),
        'chunks': len(chunks),
        'relevant_chunks': len(relevant),
        'embedding_model': embedding_model
    }

print('Pipeline functions loaded.')

## 6. Batch Execution

In [ ]:
batch_results = []

print(f'=== BATCH EVALUATION: {len(categories)} Categories ===\n')
print(f'Pipelines to run:')
print(f'  Original:              {RUN_ORIGINAL_PIPELINE}')
print(f'  Enhanced (small):      {RUN_ENHANCED_SMALL}')
print(f'  Enhanced (large):      {RUN_ENHANCED_LARGE}')
print('\n' + '='*80 + '\n')

for idx, cat in enumerate(tqdm(categories, desc='Processing categories')):
    cat_id = cat['id']
    cat_name = cat['name']
    cat_path = cat['path']
    expected = cat['expected_score']
    
    if VERBOSE:
        print(f'\n[{idx+1}/{len(categories)}] Category {cat_id}: {cat_name}')
        print(f'  Expected score: {expected}')
    
    kb = load_kb(cat_id)
    
    result = {
        'category_id': cat_id,
        'category_name': cat_name,
        'expected': expected
    }
    
    if RUN_ORIGINAL_PIPELINE:
        if VERBOSE:
            print(f'  Running original pipeline...')
        try:
            orig = run_original_pipeline(cat_id, cat_path, kb)
            result['original'] = orig
            result['original_correct'] = int(orig['predicted'] == expected)
            if VERBOSE:
                print(f'    Predicted: {orig["predicted"]} | Correct: {result["original_correct"]} | Time: {orig["time"]}s')
        except Exception as e:
            logger.error(f'Original pipeline error for {cat_id}: {e}')
            result['original'] = {'predicted': 0, 'confidence': 0.0, 'time': 0, 'error': str(e)}
            result['original_correct'] = 0
    
    if RUN_ENHANCED_SMALL:
        if VERBOSE:
            print(f'  Running enhanced pipeline (small embedding)...')
        try:
            db_path = os.path.join(DB_DIR, f'cat_{cat_id.replace(".", "_")}_small.db')
            enh_small = run_enhanced_pipeline(cat_id, cat_path, kb, EMBEDDING_MODEL_SMALL, EMBEDDING_DIM_SMALL, db_path, use_openai_agg=False)
            result['enhanced_small'] = enh_small
            result['enhanced_small_correct'] = int(enh_small['predicted'] == expected)
            if VERBOSE:
                print(f'    Predicted: {enh_small["predicted"]} | Correct: {result["enhanced_small_correct"]} | Time: {enh_small["time"]}s')
        except Exception as e:
            logger.error(f'Enhanced (small) pipeline error for {cat_id}: {e}')
            result['enhanced_small'] = {'predicted': 0, 'confidence': 0.0, 'time': 0, 'error': str(e)}
            result['enhanced_small_correct'] = 0
    
    if RUN_ENHANCED_LARGE:
        if VERBOSE:
            print(f'  Running enhanced pipeline (large embedding)...')
        try:
            db_path = os.path.join(DB_DIR, f'cat_{cat_id.replace(".", "_")}_large.db')
            enh_large = run_enhanced_pipeline(cat_id, cat_path, kb, EMBEDDING_MODEL_LARGE, EMBEDDING_DIM_LARGE, db_path, use_openai_agg=False)
            result['enhanced_large'] = enh_large
            result['enhanced_large_correct'] = int(enh_large['predicted'] == expected)
            if VERBOSE:
                print(f'    Predicted: {enh_large["predicted"]} | Correct: {result["enhanced_large_correct"]} | Time: {enh_large["time"]}s')
        except Exception as e:
            logger.error(f'Enhanced (large) pipeline error for {cat_id}: {e}')
            result['enhanced_large'] = {'predicted': 0, 'confidence': 0.0, 'time': 0, 'error': str(e)}
            result['enhanced_large_correct'] = 0
    
    batch_results.append(result)
    
    if SAVE_INTERMEDIATE_RESULTS and (idx + 1) % 5 == 0:
        interim_path = os.path.join(REPORT_DIR, f'batch_results_interim_{idx+1}.json')
        with open(interim_path, 'w', encoding='utf-8') as f:
            json.dump({'timestamp': datetime.now().isoformat(), 'results': batch_results}, f, indent=2, ensure_ascii=False)
        if VERBOSE:
            print(f'  Saved intermediate results to {interim_path}')

print('\n' + '='*80)
print(f'\nBatch evaluation complete! Processed {len(batch_results)} categories.')

## 7. Results Summary

In [ ]:
df_summary = pd.DataFrame(batch_results)

print('=== BATCH RESULTS SUMMARY ===\n')

if RUN_ORIGINAL_PIPELINE:
    orig_correct = df_summary['original_correct'].sum()
    orig_accuracy = orig_correct / len(df_summary) * 100
    orig_time = df_summary['original'].apply(lambda x: x.get('time', 0)).sum()
    print(f'Original Pipeline:')
    print(f'  Accuracy: {orig_correct}/{len(df_summary)} ({orig_accuracy:.1f}%)')
    print(f'  Total time: {orig_time:.1f}s')
    print()

if RUN_ENHANCED_SMALL:
    small_correct = df_summary['enhanced_small_correct'].sum()
    small_accuracy = small_correct / len(df_summary) * 100
    small_time = df_summary['enhanced_small'].apply(lambda x: x.get('time', 0)).sum()
    print(f'Enhanced Pipeline (text-embedding-3-small):')
    print(f'  Accuracy: {small_correct}/{len(df_summary)} ({small_accuracy:.1f}%)')
    print(f'  Total time: {small_time:.1f}s')
    print()

if RUN_ENHANCED_LARGE:
    large_correct = df_summary['enhanced_large_correct'].sum()
    large_accuracy = large_correct / len(df_summary) * 100
    large_time = df_summary['enhanced_large'].apply(lambda x: x.get('time', 0)).sum()
    print(f'Enhanced Pipeline (text-embedding-3-large):')
    print(f'  Accuracy: {large_correct}/{len(df_summary)} ({large_accuracy:.1f}%)')
    print(f'  Total time: {large_time:.1f}s')
    print()

display_cols = ['category_id', 'expected']
if RUN_ORIGINAL_PIPELINE:
    display_cols.extend(['original_correct', 'original'])
if RUN_ENHANCED_SMALL:
    display_cols.extend(['enhanced_small_correct', 'enhanced_small'])
if RUN_ENHANCED_LARGE:
    display_cols.extend(['enhanced_large_correct', 'enhanced_large'])

df_display = df_summary[display_cols].copy()

if RUN_ORIGINAL_PIPELINE:
    df_display['orig_pred'] = df_display['original'].apply(lambda x: x.get('predicted', 0))
    df_display['orig_conf'] = df_display['original'].apply(lambda x: x.get('confidence', 0.0))
    df_display['orig_time'] = df_display['original'].apply(lambda x: x.get('time', 0))

if RUN_ENHANCED_SMALL:
    df_display['small_pred'] = df_display['enhanced_small'].apply(lambda x: x.get('predicted', 0))
    df_display['small_conf'] = df_display['enhanced_small'].apply(lambda x: x.get('confidence', 0.0))
    df_display['small_time'] = df_display['enhanced_small'].apply(lambda x: x.get('time', 0))
    df_display['small_chunks'] = df_display['enhanced_small'].apply(lambda x: x.get('chunks', 0))
    df_display['small_relevant'] = df_display['enhanced_small'].apply(lambda x: x.get('relevant_chunks', 0))

if RUN_ENHANCED_LARGE:
    df_display['large_pred'] = df_display['enhanced_large'].apply(lambda x: x.get('predicted', 0))
    df_display['large_conf'] = df_display['enhanced_large'].apply(lambda x: x.get('confidence', 0.0))
    df_display['large_time'] = df_display['enhanced_large'].apply(lambda x: x.get('time', 0))
    df_display['large_chunks'] = df_display['enhanced_large'].apply(lambda x: x.get('chunks', 0))
    df_display['large_relevant'] = df_display['enhanced_large'].apply(lambda x: x.get('relevant_chunks', 0))

drop_cols = [c for c in ['original', 'enhanced_small', 'enhanced_large'] if c in df_display.columns]
df_display = df_display.drop(columns=drop_cols)

display(df_display)

## 8. Save Results

In [ ]:
final_report = {
    'timestamp': datetime.now().isoformat(),
    'total_categories': len(batch_results),
    'configuration': {
        'run_original': RUN_ORIGINAL_PIPELINE,
        'run_enhanced_small': RUN_ENHANCED_SMALL,
        'run_enhanced_large': RUN_ENHANCED_LARGE,
        'embedding_model_small': EMBEDDING_MODEL_SMALL,
        'embedding_model_large': EMBEDDING_MODEL_LARGE,
        'chunk_size': CHUNK_SIZE,
        'chunk_overlap': CHUNK_OVERLAP,
        'top_k': TOP_K
    },
    'summary': {},
    'results': batch_results
}

if RUN_ORIGINAL_PIPELINE:
    final_report['summary']['original'] = {
        'correct': int(df_summary['original_correct'].sum()),
        'accuracy': float(df_summary['original_correct'].mean() * 100),
        'total_time': float(df_summary['original'].apply(lambda x: x.get('time', 0)).sum())
    }

if RUN_ENHANCED_SMALL:
    final_report['summary']['enhanced_small'] = {
        'correct': int(df_summary['enhanced_small_correct'].sum()),
        'accuracy': float(df_summary['enhanced_small_correct'].mean() * 100),
        'total_time': float(df_summary['enhanced_small'].apply(lambda x: x.get('time', 0)).sum())
    }

if RUN_ENHANCED_LARGE:
    final_report['summary']['enhanced_large'] = {
        'correct': int(df_summary['enhanced_large_correct'].sum()),
        'accuracy': float(df_summary['enhanced_large_correct'].mean() * 100),
        'total_time': float(df_summary['enhanced_large'].apply(lambda x: x.get('time', 0)).sum())
    }

json_path = os.path.join(REPORT_DIR, 'batch_evaluation_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False)

csv_path = os.path.join(REPORT_DIR, 'batch_evaluation_results.csv')
df_display.to_csv(csv_path, index=False)

print(f'Results saved:')
print(f'  JSON: {json_path}')
print(f'  CSV:  {csv_path}')
print(f'\nJSON file size: {os.path.getsize(json_path) / 1024:.1f} KB')
print(f'CSV file size: {os.path.getsize(csv_path) / 1024:.1f} KB')

## 9. Generate HTML Dashboard

In [ ]:
def generate_dashboard_html(report_data: Dict[str, Any], output_path: str):
    json_str = json.dumps(report_data, ensure_ascii=False)
    
    html = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>CSMS AI Batch Evaluation Dashboard</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: 'Segoe UI', sans-serif; }}
        body {{ background: #0f172a; color: #e2e8f0; padding: 2rem; }}
        .container {{ max-width: 1600px; margin: 0 auto; }}
        h1 {{ font-size: 2rem; margin-bottom: 0.5rem; background: linear-gradient(135deg, #3b82f6, #10b981); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }}
        .subtitle {{ color: #94a3b8; margin-bottom: 2rem; }}
        .summary-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 1rem; margin-bottom: 2rem; }}
        .summary-card {{ background: #1e293b; padding: 1.5rem; border-radius: 12px; border: 1px solid #334155; }}
        .summary-card h3 {{ font-size: 0.85rem; color: #94a3b8; text-transform: uppercase; margin-bottom: 0.5rem; }}
        .summary-card .value {{ font-size: 2rem; font-weight: 700; }}
        .value.correct {{ color: #10b981; }}
        .value.partial {{ color: #f59e0b; }}
        .value.incorrect {{ color: #ef4444; }}
        table {{ width: 100%; border-collapse: collapse; background: #1e293b; border-radius: 12px; overflow: hidden; margin-bottom: 2rem; }}
        th, td {{ padding: 1rem; text-align: left; border-bottom: 1px solid #334155; }}
        th {{ background: #0f172a; font-weight: 600; color: #94a3b8; font-size: 0.85rem; text-transform: uppercase; }}
        tr:last-child td {{ border-bottom: none; }}
        tr:hover td {{ background: rgba(59, 130, 246, 0.05); }}
        .badge {{ padding: 0.3rem 0.7rem; border-radius: 99px; font-size: 0.75rem; font-weight: 600; display: inline-block; }}
        .badge.correct {{ background: rgba(16, 185, 129, 0.2); color: #10b981; }}
        .badge.incorrect {{ background: rgba(239, 68, 68, 0.2); color: #ef4444; }}
        .progress-bar {{ background: #334155; border-radius: 6px; height: 20px; overflow: hidden; position: relative; }}
        .progress-fill {{ height: 100%; transition: width 0.3s ease; display: flex; align-items: center; justify-content: center; font-size: 0.7rem; font-weight: 600; color: white; }}
        .progress-fill.correct {{ background: linear-gradient(90deg, #10b981, #059669); }}
        .progress-fill.partial {{ background: linear-gradient(90deg, #f59e0b, #d97706); }}
        .progress-fill.incorrect {{ background: linear-gradient(90deg, #ef4444, #dc2626); }}
        .comparison {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 0.5rem; margin-top: 0.5rem; }}
        .comparison-item {{ background: #0f172a; padding: 0.5rem; border-radius: 6px; text-align: center; font-size: 0.8rem; }}
        .comparison-item .label {{ color: #94a3b8; font-size: 0.7rem; margin-bottom: 0.2rem; }}
        .comparison-item .score {{ font-size: 1.2rem; font-weight: 700; }}
    </style>
</head>
<body>
<div class="container">
    <h1>CSMS AI Batch Evaluation Dashboard</h1>
    <p class="subtitle">Marine Category Scoring — {report_data['total_categories']} Categories Evaluated</p>
    
    <div class="summary-grid" id="summary-grid"></div>
    
    <h2 style="margin-bottom: 1rem;">Category Results</h2>
    <table id="results-table">
        <thead>
            <tr>
                <th>#</th>
                <th>Category ID</th>
                <th>Category Name</th>
                <th>Expected</th>
                <th>Original</th>
                <th>Enhanced (Small)</th>
                <th>Enhanced (Large)</th>
                <th>Best Match</th>
            </tr>
        </thead>
        <tbody id="results-tbody"></tbody>
    </table>
</div>

<script>
const DATA = {json_str};

function renderSummary() {{
    const summary = DATA.summary;
    const total = DATA.total_categories;
    let html = '';
    
    if (summary.original) {{
        html += `<div class="summary-card">
            <h3>Original Pipeline</h3>
            <div class="value">${{summary.original.correct}}/${{total}}</div>
            <div style="color: #94a3b8; font-size: 0.9rem; margin-top: 0.5rem;">${{summary.original.accuracy.toFixed(1)}}% accuracy</div>
        </div>`;
    }}
    
    if (summary.enhanced_small) {{
        html += `<div class="summary-card">
            <h3>Enhanced (Small)</h3>
            <div class="value">${{summary.enhanced_small.correct}}/${{total}}</div>
            <div style="color: #94a3b8; font-size: 0.9rem; margin-top: 0.5rem;">${{summary.enhanced_small.accuracy.toFixed(1)}}% accuracy</div>
        </div>`;
    }}
    
    if (summary.enhanced_large) {{
        html += `<div class="summary-card">
            <h3>Enhanced (Large)</h3>
            <div class="value">${{summary.enhanced_large.correct}}/${{total}}</div>
            <div style="color: #94a3b8; font-size: 0.9rem; margin-top: 0.5rem;">${{summary.enhanced_large.accuracy.toFixed(1)}}% accuracy</div>
        </div>`;
    }}
    
    document.getElementById('summary-grid').innerHTML = html;
}}

function renderResults() {{
    const results = DATA.results;
    let html = '';
    
    results.forEach((r, idx) => {{
        const expected = r.expected;
        
        let origScore = '-', origBadge = '';
        if (r.original) {{
            origScore = r.original.predicted;
            origBadge = r.original_correct ? '<span class="badge correct">Correct</span>' : '<span class="badge incorrect">Wrong</span>';
        }}
        
        let smallScore = '-', smallBadge = '';
        if (r.enhanced_small) {{
            smallScore = r.enhanced_small.predicted;
            smallBadge = r.enhanced_small_correct ? '<span class="badge correct">Correct</span>' : '<span class="badge incorrect">Wrong</span>';
        }}
        
        let largeScore = '-', largeBadge = '';
        if (r.enhanced_large) {{
            largeScore = r.enhanced_large.predicted;
            largeBadge = r.enhanced_large_correct ? '<span class="badge correct">Correct</span>' : '<span class="badge incorrect">Wrong</span>';
        }}
        
        let bestMatch = 'None';
        if (r.original_correct || r.enhanced_small_correct || r.enhanced_large_correct) {{
            const matches = [];
            if (r.original_correct) matches.push('Orig');
            if (r.enhanced_small_correct) matches.push('Small');
            if (r.enhanced_large_correct) matches.push('Large');
            bestMatch = matches.join(', ');
        }}
        
        html += `<tr>
            <td>${{idx + 1}}</td>
            <td><strong>${{r.category_id}}</strong></td>
            <td>${{r.category_name.substring(0, 50)}}${{r.category_name.length > 50 ? '...' : ''}}</td>
            <td><strong>${{expected}}</strong></td>
            <td>
                <div style="font-size: 1.2rem; font-weight: 700;">${{origScore}}</div>
                ${{origBadge}}
            </td>
            <td>
                <div style="font-size: 1.2rem; font-weight: 700;">${{smallScore}}</div>
                ${{smallBadge}}
            </td>
            <td>
                <div style="font-size: 1.2rem; font-weight: 700;">${{largeScore}}</div>
                ${{largeBadge}}
            </td>
            <td>${{bestMatch}}</td>
        </tr>`;
    }});
    
    document.getElementById('results-tbody').innerHTML = html;
}}

renderSummary();
renderResults();
</script>
</body>
</html>''';
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

dashboard_path = os.path.join(REPORT_DIR, 'batch_evaluation_dashboard.html')
generate_dashboard_html(final_report, dashboard_path)

print(f'HTML dashboard generated: {dashboard_path}')
print(f'File size: {os.path.getsize(dashboard_path) / 1024:.1f} KB')
print(f'\nOpen this file in a browser to view the interactive dashboard.')

## 10. Conclusion

### What Was Tested:
- **29 marine categories** with varying expected scores (0, 6, 10)
- **3 pipeline variants:**
  1. Original pipeline (flat aggregation)
  2. Enhanced pipeline with `text-embedding-3-small` (1536-dim)
  3. Enhanced pipeline with `text-embedding-3-large` (3072-dim)

### Key Metrics:
- **Accuracy:** Percentage of categories where predicted score matches expected
- **Processing time:** Total time for extraction + embedding + retrieval + jurors + consensus
- **Chunk statistics:** Number of chunks created and relevant chunks retrieved

### Files Generated:
1. `batch_evaluation_results.json` — Complete structured results
2. `batch_evaluation_results.csv` — Tabular summary
3. `batch_evaluation_dashboard.html` — Interactive HTML dashboard
4. Individual SQLite databases per category in `outputs/databases/`

### Next Steps:
- Analyze which pipeline variant performs best
- Investigate categories where all pipelines failed
- Tune chunk size, overlap, and top-k parameters
- Consider using `gpt-4o-mini` for enhanced aggregation summarization
- Add batch embedding cache to avoid re-embedding unchanged documents